In [2]:
import os, joblib, sklearn
import pandas as pd
from sklearn.model_selection import train_test_split
print("scikit-learn", sklearn.__version__)

raw = pd.read_csv("bank.csv", sep=";")
print(raw.shape)
print(raw["y"].value_counts())
raw["y"] = raw["y"].map({"no": 0, "yes": 1})

DROP_DURATION = False        # keep it for now; we can change it later
if DROP_DURATION:
    raw = raw.drop(columns=["duration"])

train_part, test_part = train_test_split(raw, test_size=0.2, stratify=raw["y"], random_state=7)
test_part.to_csv("test_data.csv", index=False)
X_train, y_train = train_part.drop(columns="y"), train_part["y"]
X_test,  y_test  = test_part.drop(columns="y"),  test_part["y"]

scikit-learn 1.9.1
(4521, 17)
y
no     4000
yes     521
Name: count, dtype: int64


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

num_cols = X_train.select_dtypes(include="number").columns.tolist()
cat_cols = X_train.select_dtypes(exclude="number").columns.tolist()
print(len(num_cols), "numeric,", len(cat_cols), "categorical")

def make_prep():
    return ColumnTransformer([
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ])

7 numeric, 9 categorical


In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree":       DecisionTreeClassifier(max_depth=5, random_state=7),
    "KNN":                 KNeighborsClassifier(n_neighbors=9),
    "Naive Bayes":         GaussianNB(),
    "SVM":                 SVC(kernel="rbf"),
}

def score_model(pipe, X, y):
    pred = pipe.predict(X)
    if hasattr(pipe, "predict_proba"):
        s = pipe.predict_proba(X)[:, 1]
    else:
        s = pipe.decision_function(X)      # SVM: AUC works from this score
    return {"Accuracy": accuracy_score(y, pred), "AUC": roc_auc_score(y, s),
            "Precision": precision_score(y, pred, zero_division=0),
            "Recall": recall_score(y, pred), "F1": f1_score(y, pred),
            "MCC": matthews_corrcoef(y, pred)}

In [10]:
os.makedirs("model", exist_ok=True)
rows = {}
for name, clf in models.items():
    pipe = Pipeline([("prep", make_prep()), ("clf", clf)])
    pipe.fit(X_train, y_train)
    rows[name] = score_model(pipe, X_test, y_test)
    joblib.dump(pipe, "model/" + name.replace(" ", "_").lower() + ".joblib")
    print("done:", name)

results = pd.DataFrame(rows).T.round(4)
print(results)
results.to_csv("model_results.csv")

done: Logistic Regression
done: Decision Tree
done: KNN
done: Naive Bayes
done: SVM
                     Accuracy     AUC  Precision  Recall      F1     MCC
Logistic Regression    0.8972  0.8759     0.6222  0.2692  0.3758  0.3639
Decision Tree          0.8983  0.8232     0.6034  0.3365  0.4321  0.4008
KNN                    0.8939  0.8181     0.6538  0.1635  0.2615  0.2906
Naive Bayes            0.8442  0.7734     0.3520  0.4231  0.3843  0.2976
SVM                    0.9039  0.8759     0.7742  0.2308  0.3556  0.3893
